In [16]:
# ==========================================
# PHASE 1: IMPORTS WITH COMPREHENSIVE ERROR HANDLING
# ==========================================

import os
import json
import logging
from pathlib import Path
from typing import Optional
HEADERS = {
    "x-rapidapi-key": os.getenv('RAPIDAPI_KEY', ''),
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Load environment variables with error handling
try:
    from dotenv import load_dotenv
    load_dotenv()
    logger.info("✅ Environment variables loaded from .env file")
except ImportError as e:
    logger.warning(f"⚠️ python-dotenv not installed: {e}. Using fallback environment.")
except Exception as e:
    logger.error(f"❌ Failed to load .env file: {e}")

# Initialize environment variables with fallback values
env_vars = {
    'GOOGLE_API_KEY': 'GOOGLE_API_KEY',
    'TAVILY_API_KEY': 'TAVILY_API_KEY',
    'GROQ_API_KEY': 'GROQ_API_KEY',
    'RAPIDAPI_KEY': 'RAPIDAPI_KEY',
    'WEATHER_API_KEY': 'WEATHER_API_KEY'
}

for var_name, var_key in env_vars.items():
    try:
        value = os.getenv(var_key)
        if value:
            os.environ[var_name] = value
            logger.info(f"✅ {var_name} loaded successfully")
        else:
            logger.warning(f"⚠️ {var_name} not found in environment. Using default.")
    except Exception as e:
        logger.error(f"❌ Error loading {var_name}: {e}")

# Initialize LangChain chat model with error handling
try:
    from langchain.chat_models import init_chat_model
    model = init_chat_model(model='openai/gpt-oss-120b', model_provider="groq")
    logger.info("✅ LangChain model initialized successfully (Groq)")
except ImportError as e:
    logger.error(f"❌ Failed to import init_chat_model: {e}")
    raise SystemExit("Required dependency 'langchain' not found. Please install it.")
except Exception as e:
    logger.error(f"❌ Failed to initialize chat model: {e}")
    raise SystemExit(f"Chat model initialization failed: {e}")
# model = init_chat_model(model='gemini-3-pro-preview',model_provider="google_genai",base_url="https://api.ai-gateway.tigeranalytics.com")

# ================

2026-02-23 11:09:52,282 - INFO - ✅ Environment variables loaded from .env file
2026-02-23 11:09:52,284 - INFO - ✅ GOOGLE_API_KEY loaded successfully
2026-02-23 11:09:52,285 - INFO - ✅ TAVILY_API_KEY loaded successfully
2026-02-23 11:09:52,286 - INFO - ✅ GROQ_API_KEY loaded successfully
2026-02-23 11:09:52,286 - INFO - ✅ RAPIDAPI_KEY loaded successfully
2026-02-23 11:09:52,287 - INFO - ✅ WEATHER_API_KEY loaded successfully
2026-02-23 11:09:52,443 - INFO - ✅ LangChain model initialized successfully (Groq)


In [17]:
import json
import re
import requests
from langchain.tools import tool

@tool
def fetch_match_stats(match_id: str) -> dict:
    """
    Fetches raw scorecard data from the API and organizes it into 
    batting and bowling maps for calculation.
    """
    url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}/scard"
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        data = response.json()
        performance_map = {}

        for innings in data.get('scorecard', []):
            # --- 1. PARSE BATTING ---
            for bat in innings.get('batsman', []):
                p_id = str(bat['id'])
                
                # Logic: Award LBW/Bowled bonus if found in outdec string
                out_desc = bat.get('outdec', "").lower()
                lbw_bowled = 1 if ("lbw" in out_desc or out_desc.startswith("b ")) else 0

                bat_stats = {
                    "name": bat.get('name'),
                    "batting": {
                        "runs": int(bat.get('runs', 0)),
                        "balls": int(bat.get('balls', 0)),
                        "fours": int(bat.get('fours', 0)),
                        "sixes": int(bat.get('sixes', 0)),
                        "sr": float(bat.get('strkrate', 0)),
                        "is_out": out_desc != "not out",
                        "lbw_bowled_victim": lbw_bowled # Useful for bowler bonus later
                    }
                }
                
                if p_id not in performance_map:
                    performance_map[p_id] = bat_stats
                else:
                    performance_map[p_id]["batting"] = bat_stats["batting"]

            # --- 2. PARSE BOWLING ---
            for bowl in innings.get('bowler', []):
                p_id = str(bowl['id'])
                
                # Logic: Count how many times this bowler bowled someone out or got LBW
                # (This requires looking at the batting list of the OPPOSING innings)
                # For simplicity here, we extract direct bowler stats first:
                bowl_stats = {
                    "bowling": {
                        "wickets": int(bowl.get('wickets', 0)),
                        "overs": float(bowl.get('overs', 0)),
                        "maidens": int(bowl.get('maidens', 0)),
                        "runs_conceded": int(bowl.get('runs', 0)),
                        "econ": float(bowl.get('economy', 0)),
                        "dots": int(bowl.get('dots', 0))
                    }
                }

                if p_id not in performance_map:
                    performance_map[p_id] = {"name": bowl.get('name'), "bowling": bowl_stats["bowling"]}
                else:
                    performance_map[p_id]["bowling"] = bowl_stats["bowling"]

        return performance_map
    except Exception as e:
        return {"error": str(e)}
def calculate_fantasy_points_2026(p_data):
    """
    Standard T20 Fantasy Points (2026 Edition).
    Handles nested 'batting' and 'bowling' keys from our performance map.
    """
    pts = 0.0  # Use float for precision, cast to int at end
    
    # --- 1. BATTING SECTION ---
    if "batting" in p_data:
        bat = p_data["batting"]
        runs = int(bat.get('runs', 0))
        balls = int(bat.get('balls', 0))
        fours = int(bat.get('fours', 0))
        sixes = int(bat.get('sixes', 0))
        sr = float(bat.get('sr', 0))

        # Base Points
        pts += runs 
        pts += (fours * 1)    # Boundary Bonus
        pts += (sixes * 2)    # Six Bonus
        
        # Milestones
        if runs >= 100: pts += 16
        elif runs >= 50: pts += 8
        elif runs >= 30: pts += 4
        
        # Duck Penalty (Excludes Bowlers who didn't score)
        if runs == 0 and balls > 0:
            pts -= 2

        # Strike Rate Bonus (Min 10 balls)
        if balls >= 10:
            if sr > 170: pts += 6
            elif 150 < sr <= 170: pts += 4
            elif 130 <= sr <= 150: pts += 2
            elif 60 <= sr <= 70: pts -= 2
            elif 50 <= sr < 60: pts -= 4
            elif sr < 50: pts -= 6

    # --- 2. BOWLING SECTION ---
    if "bowling" in p_data:
        bowl = p_data["bowling"]
        wickets = int(bowl.get('wickets', 0))
        overs = float(bowl.get('overs', 0))
        econ = float(bowl.get('econ', 0))
        maidens = int(bowl.get('maidens', 0))

        # Base Wicket Points
        pts += (wickets * 25)
        
        # LBW / Bowled Bonus (+8 per wicket)
        # We use the count we extracted from the 'outdec' string
        lbw_bowled_count = int(bowl.get('lbw_bowled_count', 0))
        pts += (lbw_bowled_count * 8)
        
        # Wicket Milestones
        if wickets >= 5: pts += 16
        elif wickets >= 4: pts += 8
        elif wickets >= 3: pts += 4
        
        # Maiden Over Bonus
        pts += (maidens * 12)

        # Economy Rate Bonus (Min 2 Overs)
        if overs >= 2.0:
            if econ < 5: pts += 6
            elif 5 <= econ < 6: pts += 4
            elif 6 <= econ <= 7: pts += 2
            elif 10 <= econ < 11: pts -= 2
            elif 11 <= econ <= 12: pts -= 4
            elif econ > 12: pts -= 6

    return int(pts)
@tool
def get_user_selections(file_path) -> list:
    """
    Reads the nightly log file and extracts User names, Agent types, 
    and Player IDs from the markdown tables.
    """
    try:
        with open(file_path, "r") as f:
            logs = json.load(f)
        
        final_teams = []
        for entry in logs:
            raw_text = entry.get("team", "")
            player_matches = re.findall(r"\|\s*(\d+)\s*\|\s*([^|]+?)\s*\|", raw_text)
            
            players = [{"id": p_id, "name": p_name.strip()} for p_id, p_name in player_matches]
            final_teams.append({
                "user": entry.get("user"),
                "agent": entry.get("agent_type"),
                "players": players
            })
        return final_teams
    except Exception as e:
        return [{"error": str(e)}]

@tool
def calculate_leaderboard(match_id: str) -> str:
    """
    The master tool: Fetches stats, reads user logs, calculates points 
    using 2026 Fantasy Rules, and returns a formatted Leaderboard string.
    """
    stats = fetch_match_stats.run(match_id)
    print('inside')
    teams = get_user_selections.run("/mnt/c/workspaces/agent_project/notebooks1/agent_selections_log.json")
    
    if "error" in stats: return f"Error fetching stats: {stats['error']}"

    results = []
    for team in teams:
        score = 0
        for p in team['players']:
            p_stats = stats.get(str(p['id']), {})
            score += calculate_fantasy_points_2026(p_stats) # Using your logic function
        
        results.append({"user": team['user'], "agent": team['agent'], "score": score})

    # Sorting
    results.sort(key=lambda x: x['score'], reverse=True)
    
    # Formatting
    output = "\n" + "═"*55 + "\n"
    output += f"🏆  FANTASY LEADERBOARD (Match ID: {match_id})  🏆\n"
    output += "═"*55 + "\n"
    output += f"{'Rank':<5} | {'User Name':<15} | {'Agent':<12} | {'Score':<8}\n"
    output += "─"*55 + "\n"
    for i, res in enumerate(results, 1):
        output += f"{i:<5} | {res['user']:<15} | {res['agent']:<12} | {res['score']:<8}\n"
    output += "═"*55
    return output

In [18]:
validator_prompt = """You are the "Fantasy Audit Chief". 
Your sole purpose is to validate the performance of our scouting agents (Form-Scout and Eco-Scout).

When a user provides a match_id:
1. Use the 'calculate_leaderboard' tool to process the results.
2. Present the final rankings clearly.
3. Provide a brief insight on which agent (Form or Eco) won the night.

STRICT RULES:
- Only respond with the leaderboard and a small summary.
- Do not make up player stats.
- If the leaderboard tool fails, report the error.
"""
from langchain.agents import create_agent
validator_agent = create_agent(
    model=model,
    tools=[calculate_leaderboard],
    system_prompt=validator_prompt
)

In [19]:
# Final Execution
user_input = "Validate the performance for match_id 139351."

for chunk in validator_agent.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    stream_mode="updates",
):
    for node_name, data in chunk.items():
        new_msg = data['messages'][-1]
        if hasattr(new_msg, 'content') and new_msg.content:
            print(new_msg.content)

2026-02-23 11:10:07,408 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


inside

═══════════════════════════════════════════════════════
🏆  FANTASY LEADERBOARD (Match ID: 139351)  🏆
═══════════════════════════════════════════════════════
Rank  | User Name       | Agent        | Score   
───────────────────────────────────────────────────────
1     | Player_3        | Form-Scout   | 435     
2     | Player_1        | Form-Scout   | 265     
3     | Player_2        | Form-Scout   | 241     
4     | Player_4        | Eco-Scout    | 0       
5     | Player_5        | Eco-Scout    | 0       
═══════════════════════════════════════════════════════


2026-02-23 11:10:09,385 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


**Fantasy Leaderboard – Match 139351**

| Rank | User     | Agent      | Score |
|------|----------|------------|-------|
| 1    | Player 3 | Form‑Scout | 435 |
| 2    | Player 1 | Form‑Scout | 265 |
| 3    | Player 2 | Form‑Scout | 241 |
| 4    | Player 4 | Eco‑Scout  | 0 |
| 5    | Player 5 | Eco‑Scout  | 0 |

**Summary:**  
Form‑Scout dominated the night, capturing the top three spots and the highest scores. Eco‑Scout did not register any points, so Form‑Scout is the clear winner for this match.
